In [9]:
import pandas as pd
import numpy as np

from src.testing_validation.model_test import calculate_rmse
from src.helpers import fit_cv_timeseries_model

from lightgbm import LGBMRegressor

All Data

In [10]:

combined_data = (
    pd.read_parquet("../data/train.parquet")
    .sort_values("date")
    .reset_index(drop=True)
)

combined_data["usd_zar_28_movement"] = (
    combined_data["usd_zar_28"] - combined_data["usd_zar"]
)
combined_data.head(1)

,date,gold_usd_per_oz,platinum_usd_per_oz,gold_usd_per_oz_return,platinum_usd_per_oz_return,us_fed_funds,us_5y_yield,vix,broad_usd_index,iron_ore_usd_per_tonne,...,usd_zar,usd_zar_28,usd_zar_1w_return,usd_zar_1m_return,usd_zar_3m_return,usd_zar_1m_volatility,richards_bay_coal_usd,sa_5y_cds_bp,sa_5y_yield,usd_zar_28_movement
0,2008-10-10,855.400024,996.700012,-0.019289,-0.007625,0.79,2.77,69.95,97.999,60.8,...,9.3626,10.0284,0.085265,0.066904,-0.004945,0.043376,112.4,455.4,9.115,0.6658


In [11]:
X_all = combined_data.drop(columns=["date", "usd_zar", "usd_zar_28_movement"])

In [12]:
y = combined_data["usd_zar_28_movement"]

In [13]:
model = LGBMRegressor()
fit_cv_timeseries_model(model, X_all, y)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000374 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2909
[LightGBM] [Info] Number of data points in the train set: 535, number of used features: 22
[LightGBM] [Info] Start training from score -0.118947
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, 

np.float64(0.6025146828137833)

Engineered data

In [14]:

engineered_data = combined_data.copy().drop(columns=["date", "usd_zar_28", "usd_zar_28_movement"])
engineered_data["interest_rate_diff"] = (
    engineered_data["sa_repo_rate"] - engineered_data["us_fed_funds"]
)
engineered_data["sa_us_5y_yield_spread"] = (
    engineered_data["sa_5y_yield"] - engineered_data["us_5y_yield"]
)

engineered_data["commodities"] = np.mean([engineered_data.iron_ore_usd_per_tonne, 
engineered_data.gold_usd_per_oz, engineered_data.platinum_usd_per_oz, engineered_data.richards_bay_coal_usd], axis=0)

engineered_features = engineered_data.columns
# [
#     #"commodities",
#     #"usd_zar",
#     #"gold_usd_per_oz",
#     #"platinum_usd_per_oz",
#     #"richards_bay_coal_usd",
#     #"iron_ore_usd_per_tonne",
#     "brent_usd_per_barrel",
#     "interest_rate_diff",
#     "sa_us_5y_yield_spread",
#     "sa_yoy_inflation",
#     "sa_5y_cds_bp",
#     "vix",
#     "broad_usd_index",
#     "sa_cpi",
#     #"gold_usd_per_oz_return",
#     #"platinum_usd_per_oz_return",
#     #"usd_zar_1w_return",
#     #"usd_zar_1m_return",
#     #"usd_zar_3m_return",
#     #"usd_zar_1m_volatility",
# ]


In [16]:

X_engineered = engineered_data[engineered_features]
assert X_engineered.select_dtypes(exclude="number").empty

In [17]:
model = LGBMRegressor()
fit_cv_timeseries_model(model, X_engineered, y)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000186 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3335
[LightGBM] [Info] Number of data points in the train set: 535, number of used features: 25
[LightGBM] [Info] Start training from score -0.118947
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, 

np.float64(0.7272912203708323)

Backward stepwise feature selection

In [18]:
import time

from joblib import Parallel, delayed, parallel_config
from sklearn.base import clone
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import TimeSeriesSplit


def _progress(message):
    """Print immediately in Jupyter instead of waiting for a logging buffer."""
    print(f"{time.strftime('%H:%M:%S')} | {message}", flush=True)


def backward_stepwise_selection(
    estimator,
    X,
    y,
    n_splits=5,
    n_jobs=4,
    log_every=None,
    stop_when_no_improvement=False,
):
    """Evaluate feature removals along a backward-elimination path.

    By default every round removes the candidate producing the lowest CV RMSE,
    even when RMSE worsens. Set ``stop_when_no_improvement=True`` for standard
    backward elimination that stops at the first non-improving round.
    """
    selected_features = list(X.columns)
    splits = list(TimeSeriesSplit(n_splits=n_splits).split(X))
    total_start = time.perf_counter()
    n_jobs = max(1, min(n_jobs, len(selected_features)))

    base_estimator = clone(estimator)
    if "n_jobs" in base_estimator.get_params(deep=False):
        base_estimator.set_params(n_jobs=1)

    def cv_rmse(features, verbose=False):
        fold_rmses = []
        for fold, (train_idx, test_idx) in enumerate(splits, start=1):
            if features:
                fold_model = clone(base_estimator)
                fold_model.fit(X.iloc[train_idx][features], y.iloc[train_idx])
                predictions = fold_model.predict(X.iloc[test_idx][features])
            else:
                # LightGBM cannot fit an empty matrix. Use the honest
                # training-fold mean as the zero-feature comparator.
                predictions = np.full(len(test_idx), y.iloc[train_idx].mean())
            fold_rmses.append(root_mean_squared_error(y.iloc[test_idx], predictions))
            if verbose:
                _progress(
                    f"Baseline fold {fold}/{len(splits)} complete; "
                    f"RMSE={fold_rmses[-1]:.6f}"
                )
        return float(np.mean(fold_rmses))

    def evaluate_candidate(feature, remaining):
        return cv_rmse(remaining), feature

    _progress(
        f"Starting backward selection with {len(selected_features)} features, "
        f"{n_splits} folds and {n_jobs} candidate workers"
    )
    _progress("Fitting baseline CV now...")
    current_rmse = cv_rmse(selected_features, verbose=True)
    best_rmse = current_rmse
    best_features = selected_features.copy()
    history = [{
        "step": 0,
        "removed": None,
        "n_features": len(selected_features),
        "remaining_features": tuple(selected_features),
        "cv_rmse": current_rmse,
        "change_from_previous": np.nan,
        "best_so_far": True,
        "round_seconds": 0.0,
        "elapsed_seconds": time.perf_counter() - total_start,
    }]
    _progress(f"Baseline complete; mean CV RMSE={current_rmse:.6f}")

    step = 0
    while selected_features:
        step += 1
        round_start = time.perf_counter()
        candidates = [
            (feature, [name for name in selected_features if name != feature])
            for feature in selected_features
        ]
        report_every = log_every or max(1, len(candidates) // 5)
        _progress(
            f"Round {step}: evaluating {len(candidates)} removals "
            f"with {n_jobs} workers"
        )

        completed_results = []
        # Separate processes plus one inner thread prevent LightGBM/BLAS
        # oversubscription from freezing the notebook UI.
        with parallel_config(
            backend="loky",
            n_jobs=n_jobs,
            inner_max_num_threads=1,
        ):
            result_generator = Parallel(return_as="generator_unordered")(
                delayed(evaluate_candidate)(feature, remaining)
                for feature, remaining in candidates
            )
            for completed, result in enumerate(result_generator, start=1):
                completed_results.append(result)
                if completed % report_every == 0 or completed == len(candidates):
                    _progress(
                        f"Round {step}: {completed}/{len(candidates)} candidates "
                        f"complete ({time.perf_counter() - round_start:.1f}s)"
                    )

        candidate_rmse, feature_to_remove = min(completed_results)
        if stop_when_no_improvement and candidate_rmse >= current_rmse:
            _progress(
                f"Round {step}: stopping because the best removal would worsen "
                f"RMSE from {current_rmse:.6f} to {candidate_rmse:.6f}"
            )
            break

        previous_rmse = current_rmse
        selected_features.remove(feature_to_remove)
        current_rmse = candidate_rmse
        is_best = current_rmse < best_rmse
        if is_best:
            best_rmse = current_rmse
            best_features = selected_features.copy()

        history.append({
            "step": step,
            "removed": feature_to_remove,
            "n_features": len(selected_features),
            "remaining_features": tuple(selected_features),
            "cv_rmse": current_rmse,
            "change_from_previous": current_rmse - previous_rmse,
            "best_so_far": is_best,
            "round_seconds": time.perf_counter() - round_start,
            "elapsed_seconds": time.perf_counter() - total_start,
        })
        _progress(
            f"Round {step} complete: removed '{feature_to_remove}'; "
            f"{len(selected_features)} remain; RMSE={current_rmse:.6f}"
        )

    _progress(
        f"Selection finished in {time.perf_counter() - total_start:.1f}s; "
        f"best subset has {len(best_features)} features and RMSE={best_rmse:.6f}"
    )
    return best_features, pd.DataFrame(history)

In [19]:
selection_model = LGBMRegressor( n_estimators=150, learning_rate=0.05, num_leaves=15, random_state=42,
    verbosity=-1,
    n_jobs=1,
)

print("Starting exhaustive boosting elimination...", flush=True)
best_features, selection_history = backward_stepwise_selection(
    selection_model,
    X_engineered,
    y,
    n_splits=5,
    n_jobs=4,
)

best_row = selection_history.loc[selection_history["cv_rmse"].idxmin()]
print(
    f"Best subset: {len(best_features)} of {X_engineered.shape[1]} features; "
    f"CV RMSE={best_row['cv_rmse']:.6f}",
    flush=True,
)
print(best_features)
selection_history

Starting exhaustive boosting elimination...
18:51:14 | Starting backward selection with 25 features, 5 folds and 4 candidate workers
18:51:14 | Fitting baseline CV now...
18:51:15 | Baseline fold 1/5 complete; RMSE=0.392135
18:51:15 | Baseline fold 2/5 complete; RMSE=0.525069
18:51:15 | Baseline fold 3/5 complete; RMSE=0.921886
18:51:15 | Baseline fold 4/5 complete; RMSE=0.707231
18:51:15 | Baseline fold 5/5 complete; RMSE=0.983581
18:51:15 | Baseline complete; mean CV RMSE=0.705981
18:51:15 | Round 1: evaluating 25 removals with 4 workers
18:51:17 | Round 1: 5/25 candidates complete (2.0s)
18:51:17 | Round 1: 10/25 candidates complete (2.6s)
18:51:18 | Round 1: 15/25 candidates complete (3.1s)
18:51:19 | Round 1: 20/25 candidates complete (3.7s)
18:51:19 | Round 1: 25/25 candidates complete (4.4s)
18:51:19 | Round 1 complete: removed 'usd_zar'; 24 remain; RMSE=0.643301
18:51:19 | Round 2: evaluating 24 removals with 4 workers
18:51:20 | Round 2: 4/24 candidates complete (0.5s)
18:51:2

,step,removed,n_features,remaining_features,cv_rmse,change_from_previous,best_so_far,round_seconds,elapsed_seconds
0,0,NaN,25,"(gold_usd_per_oz, platinum_usd_per_oz, gold_us...",0.705981,NaN,True,0.000000,0.374619
1,1,usd_zar,24,"(gold_usd_per_oz, platinum_usd_per_oz, gold_us...",0.643301,-0.062680,True,4.379113,4.754107
2,2,iron_ore_usd_per_tonne,23,"(gold_usd_per_oz, platinum_usd_per_oz, gold_us...",0.615165,-0.028136,True,2.658977,7.413465
3,3,sa_us_5y_yield_spread,22,"(gold_usd_per_oz, platinum_usd_per_oz, gold_us...",0.593351,-0.021814,True,2.547145,9.960994
4,4,sa_real_gdp,21,"(gold_usd_per_oz, platinum_usd_per_oz, gold_us...",0.589056,-0.004294,True,2.418923,12.380275
5,5,sa_cpi,20,"(gold_usd_per_oz, platinum_usd_per_oz, gold_us...",0.575486,-0.013570,True,2.312873,14.693645
6,6,usd_zar_1m_return,19,"(gold_usd_per_oz, platinum_usd_per_oz, gold_us...",0.561405,-0.014081,True,2.080973,16.774974
7,7,gold_usd_per_oz_return,18,"(gold_usd_per_oz, platinum_usd_per_oz, platinu...",0.557411,-0.003994,True,1.894591,18.670306
8,8,sa_repo_rate,17,"(gold_usd_per_oz, platinum_usd_per_oz, platinu...",0.556964,-0.000446,True,1.740960,20.411611
9,9,sa_5y_yield,16,"(gold_usd_per_oz, platinum_usd_per_oz, platinu...",0.561029,0.004065,False,1.648708,22.060635


Regularized XGBoost using the best subset

The complete elimination path includes the zero-feature training-mean baseline. If that baseline wins, the final evaluation uses `DummyRegressor`.

In [20]:
from sklearn.dummy import DummyRegressor
from xgboost import XGBRegressor


if best_features:
    regularized_xgb_model = XGBRegressor(
        objective="reg:squarederror",
        n_estimators=500,
        learning_rate=0.03,
        max_depth=3,
        min_child_weight=5,
        gamma=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.5,
        reg_lambda=2.0,
        random_state=42,
        n_jobs=-1,
    )
    final_X = X_engineered[best_features]
else:
    print("The zero-feature baseline won; evaluating a training-mean model.")
    regularized_xgb_model = DummyRegressor(strategy="mean")
    final_X = pd.DataFrame(
        {"constant": np.ones(len(X_engineered))},
        index=X_engineered.index,
    )

fit_cv_timeseries_model(regularized_xgb_model, final_X, y)

Fold 0, rmse: 0.305256632245505
Fold 1, rmse: 0.3244946221076086
Fold 2, rmse: 0.590005174407873
Fold 3, rmse: 0.5533552526907793
Fold 4, rmse: 0.7640005261721498
RMSEs over all splits: [0.305256632245505, 0.3244946221076086, 0.590005174407873, 0.5533552526907793, 0.7640005261721498]
Mean RMSE over all folds: 0.5074224415247831


np.float64(0.5074224415247831)

Random Forest

In [21]:
from sklearn.ensemble import RandomForestRegressor


selection_model = RandomForestRegressor(
    n_estimators=50,
    max_depth=10,
    min_samples_leaf=5,
    max_features="sqrt",
    random_state=42,
    n_jobs=1,
)

print("Starting Random Forest backward elimination...", flush=True)
best_features, selection_history = backward_stepwise_selection(
    selection_model,
    X_engineered,
    y,
    n_splits=5,
    n_jobs=4,
    stop_when_no_improvement=True,
)

best_row = selection_history.loc[selection_history["cv_rmse"].idxmin()]
print(
    f"Best subset: {len(best_features)} of {X_engineered.shape[1]} features; "
    f"CV RMSE={best_row['cv_rmse']:.6f}",
    flush=True,
)
print(best_features)
selection_history

Starting Random Forest backward elimination...
18:51:47 | Starting backward selection with 25 features, 5 folds and 4 candidate workers
18:51:47 | Fitting baseline CV now...
18:51:47 | Baseline fold 1/5 complete; RMSE=0.321497
18:51:47 | Baseline fold 2/5 complete; RMSE=0.386027
18:51:47 | Baseline fold 3/5 complete; RMSE=0.756375
18:51:47 | Baseline fold 4/5 complete; RMSE=0.625768
18:51:47 | Baseline fold 5/5 complete; RMSE=0.802190
18:51:47 | Baseline complete; mean CV RMSE=0.578371
18:51:47 | Round 1: evaluating 25 removals with 4 workers
18:51:49 | Round 1: 5/25 candidates complete (1.3s)
18:51:49 | Round 1: 10/25 candidates complete (2.0s)
18:51:50 | Round 1: 15/25 candidates complete (2.7s)
18:51:51 | Round 1: 20/25 candidates complete (3.4s)
18:51:52 | Round 1: 25/25 candidates complete (4.5s)
18:51:52 | Round 1 complete: removed 'gold_usd_per_oz'; 24 remain; RMSE=0.550499
18:51:52 | Round 2: evaluating 24 removals with 4 workers
18:51:53 | Round 2: 4/24 candidates complete (0.

,step,removed,n_features,remaining_features,cv_rmse,change_from_previous,best_so_far,round_seconds,elapsed_seconds
0,0,NaN,25,"(gold_usd_per_oz, platinum_usd_per_oz, gold_us...",0.578371,NaN,True,0.000000,0.603864
1,1,gold_usd_per_oz,24,"(platinum_usd_per_oz, gold_usd_per_oz_return, ...",0.550499,-0.027873,True,4.511008,5.115304
